[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_08/listing_8.1.ipynb)

In [1]:
import sys
if 'google.colab' in sys.modules:
    pass
    # !pip install evaluate

### Listing 6.10: Environment Setup and Hardware Configuration for Unsloth

In [2]:
import os
from unsloth import FastLanguageModel
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

max_seq_length = 1024

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Listing 6.11: Loading the Model and Configuring Optimized LoRA Adapters

In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-mini-instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


==((====))==  Unsloth 2026.7.6: Fast Phi3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


### Listing 6.12: Loading, Shuffling, and Splitting the Medical QA Dataset

In [4]:
DATASET_ID = "lmassaron/medical-cardiology-qa"
print(f"Loading {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train")

n = len(dataset)
rng = np.random.default_rng(42)
all_idx = rng.permutation(n)
cut = int(n * 0.9)
train_idx, eval_idx = all_idx[:cut], all_idx[cut:]

train_ds = dataset.select(train_idx)
eval_ds = dataset.select(eval_idx)
print(f"Train samples: {len(train_ds)} | Eval samples: {len(eval_ds)}")

print("Sample message sequence:", train_ds[0]["messages"])

Loading lmassaron/medical-cardiology-qa...
Train samples: 5149 | Eval samples: 573
Sample message sequence: [{'content': 'You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology.', 'role': 'system'}, {'content': 'What is the mechanism behind the twisting pattern seen in ECG recordings of torsades de pointes?', 'role': 'user'}, {'content': 'The twisting pattern observed in ECG recordings of torsades de pointes is due to the meandering spiral wave formed by the re-entrant circuit, which bends around areas of block in the heart muscle, leading to the characteristic twisting appearance.', 'role': 'assistant'}]


### Listing 6.13: Batch Formatting Prompts with the Tokenizer Chat Template

In [5]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}


train_mapped = train_ds.map(formatting_prompts_func, batched=True)
eval_mapped = eval_ds.map(formatting_prompts_func, batched=True)
print("Prompt Preview:\n", train_mapped[0]["text"])

Prompt Preview:
 <|system|>You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology.<|end|><|user|>What is the mechanism behind the twisting pattern seen in ECG recordings of torsades de pointes?<|end|><|assistant|>The twisting pattern observed in ECG recordings of torsades de pointes is due to the meandering spiral wave formed by the re-entrant circuit, which bends around areas of block in the heart muscle, leading to the characteristic twisting appearance.<|end|>


### Listing 6.14: Configuring the SFTTrainer and Executing the Unsloth Training Loop

In [6]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=0,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=(model.dtype == torch.float16),
        bf16=(model.dtype == torch.bfloat16),
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="epoch",
        save_total_limit=1,
        output_dir="unsloth-medical-model",
        report_to="none",
        dataset_kwargs={
            "add_special_tokens": False,
        },
    ),
)

model.config.use_cache = False
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,149 | Num Epochs = 1 | Total steps = 644
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,912,896 of 3,844,934,656 (0.23% trained)


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Step,Training Loss,Validation Loss
100,0.844245,0.834720
200,0.833563,0.823818
300,0.826962,0.816237
400,0.826029,0.810690
500,0.818423,0.805763
600,0.820370,0.803366
644,0.820370,0.803142


Unsloth: Restored added_tokens_decoder metadata in unsloth-medical-model/checkpoint-644/tokenizer_config.json.


### Listing 6.15: Saving the Fine-Tuned LoRA Adapter and Tokenizer

In [7]:
model.save_pretrained("unsloth-medical-adapter")
tokenizer.save_pretrained("unsloth-medical-adapter")

Unsloth: Restored added_tokens_decoder metadata in unsloth-medical-adapter/tokenizer_config.json.


('unsloth-medical-adapter/tokenizer_config.json',
 'unsloth-medical-adapter/chat_template.jinja',
 'unsloth-medical-adapter/tokenizer.json')

### Listing 6.16: Accelerated Inference and Perplexity Evaluation with Unsloth

In [8]:
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None

eval_results = []
num_eval_samples = min(20, len(eval_ds))
print(
    f"Calculating perplexity and generating answers for {num_eval_samples} evaluation examples..."
)

for i in tqdm(range(num_eval_samples)):
    messages = eval_ds[i]["messages"]
    user_question = next(m["content"] for m in messages if m["role"] == "user")
    expected_answer = next(m["content"] for m in messages if m["role"] == "assistant")

    encoded = tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=encoded,
            max_new_tokens=256,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        outputs[0][encoded.shape[1] :], skip_special_tokens=True
    ).strip()

    ref_ids = tokenizer(
        expected_answer, return_tensors="pt", add_special_tokens=False
    ).input_ids.to("cuda")
    full_ids = torch.cat([encoded, ref_ids], dim=1)
    labels = full_ids.clone()
    labels[:, : encoded.shape[1]] = -100

    with torch.no_grad():
        outputs_loss = model(full_ids, labels=labels)
        perplexity = torch.exp(outputs_loss.loss).item()
        with model.disable_adapter():
            base_outputs_loss = model(full_ids, labels=labels)
            untrained_perplexity = torch.exp(base_outputs_loss.loss).item()

    eval_results.append(
        {
            "question": user_question,
            "expected": expected_answer,
            "generated": generated,
            "perplexity": perplexity,
            "untrained_perplexity": untrained_perplexity,
        }
    )

eval_df = pd.DataFrame(eval_results)
print(f"\nAverage Evaluation Perplexity: {eval_df['perplexity'].mean():.4f}")
print(f"Average Untrained Perplexity:  {eval_df['untrained_perplexity'].mean():.4f}")

print("\n--- Sample Comparison ---")
print("Question:", eval_df.iloc[0]["question"])
print("\nExpected Answer:", eval_df.iloc[0]["expected"])
print("\nGenerated Answer:", eval_df.iloc[0]["generated"])
print(f"Perplexity: {eval_df.iloc[0]['perplexity']:.4f}")
print(f"Untrained Perplexity:  {eval_df.iloc[0]['untrained_perplexity']:.4f}")

Calculating perplexity and generating answers for 20 evaluation examples...


100%|██████████| 20/20 [00:22<00:00,  1.11s/it]


Average Evaluation Perplexity: 4.3028
Average Untrained Perplexity:  6.0805

--- Sample Comparison ---
Question: What are some connective tissue disorders that increase the risk of aortic dissection?

Expected Answer: Connective tissue disorders that increase the risk of aortic dissection include Marfan syndrome, Ehlers-Danlos syndrome, and Loeys-Dietz syndrome. These conditions affect the structural integrity of the arterial walls.

Generated Answer: Connective tissue disorders such as Marfan syndrome, Ehlers-Danlos syndrome, and Loeys-Dietz syndrome are known to increase the risk of aortic dissection due to their impact on the structural integrity of the aortic wall.
Perplexity: 1.4326
Untrained Perplexity:  1.7253
